In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "src").exists():
    project_root = cwd
elif (cwd.parent / "src").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Could not locate project root containing 'src'.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [2]:
import numpy as np
import pandas as pd
from IPython.display import display
import statsmodels.api as sm


In [ ]:

from src.data.create_dataframes import processed_df_map

pd.set_option("display.float_format", "{:,.4f}".format)

DEFAULT_Y_COL_MAP = {
    "eur_df_processed": "EURUSD",
    "gbp_df_processed": "GBPUSD",
    "jpy_df_processed": "USDJPY",
    "chf_df_processed": "USDCHF",
    "cad_df_processed": "USDCAD",
    "aud_df_processed": "AUDUSD",
    "nzd_df_processed": "NZDUSD",
    "nok_df_processed": "USDNOK",
    "sek_df_processed": "USDSEK",
}

def latest_window_multivariate_ols(df: pd.DataFrame, y_col: str, window: int = 252):
    """Fit multivariate OLS on the most recent rolling window and return (summary, r2)."""
    if y_col not in df.columns:
        raise KeyError(f"{y_col} not found in dataframe columns.")

    window_df = df.tail(window).dropna()
    if len(window_df) < 2:
        raise ValueError("Not enough non-NaN rows in the latest rolling window.")

    y = window_df[y_col]
    X = window_df.drop(columns=[y_col])
    X_const = sm.add_constant(X, has_constant="add")

    model = sm.OLS(y, X_const).fit()
    r2 = model.rsquared

    betas = model.params.drop("const")
    pvals = model.pvalues.drop("const")
    signif_pct = (1 - pvals) * 100

    ranked_idx = betas.abs().sort_values(ascending=False).index
    betas = betas.reindex(ranked_idx)
    signif_pct = signif_pct.reindex(ranked_idx)

    summary = pd.DataFrame({
        "Driver Name": betas.index,
        "Beta Coefficient": betas.values,
        "Rank": np.arange(1, len(betas) + 1),
        "Significance %": signif_pct.values,
    })
    summary["Beta Coefficient"] = summary["Beta Coefficient"].round(4)
    summary["Significance %"] = summary["Significance %"].round(2)
    return summary, r2

def run_processed_df_regression(df_name: str, y_col: str | None = None, window: int = 252):
    """Convenience wrapper for processed_df_map by name."""
    if df_name not in processed_df_map:
        raise KeyError(f"{df_name} not found in processed_df_map.")

    if y_col is None:
        y_col = DEFAULT_Y_COL_MAP.get(df_name)
        if y_col is None:
            raise KeyError(f"No default y_col found for {df_name}.")

    summary, r2 = latest_window_multivariate_ols(processed_df_map[df_name], y_col, window=window)
    display(summary)
    print(f"Current 1Y Rolling R2: {r2:.4f}")
    return summary, r2




In [ ]:
sorted(processed_df_map.keys())


In [ ]:
# Example usage (default y_col is picked automatically)
run_processed_df_regression("nok_df_processed")


In [ ]:
def run_processed_df_regression_sig_only(
    df_name: str,
    y_col: str | None = None,
    window: int = 252,
    min_significance: float = 95.0,
):
    """Same as run_processed_df_regression but filters to >= min_significance."""
    summary, r2 = run_processed_df_regression(df_name, y_col=y_col, window=window)
    filtered = summary[summary["Significance %"] >= min_significance].reset_index(drop=True)
    display(filtered)
    print(f"Current 1Y Rolling R2: {r2:.4f}")
    return filtered, r2

run_processed_df_regression_sig_only("aud_df_processed")



In [ ]:
from src.data.create_dataframes import build_df2_map, frames, fx_newdf

# Assume you have frames and fx_newdf already loaded
df2_map = build_df2_map(frames, fx_newdf)

# To get the dataframe:
nok_df2 = df2_map["nok_df2"]
aud_df2 = df2_map["aud_df2"]
eur_df2 = df2_map["eur_df2"]


In [ ]:
aud_df2.tail()

In [ ]:
from src.data.build_ultimate_df import build_ultimate_df

ultimate_df = build_ultimate_df()
ultimate_df["aud"].tail()


In [ ]:
nok_ultimate_df = ultimate_df["nok"]
nok_ultimate_df

In [ ]:
sek_ultimate_df = ultimate_df["sek"]
sek_ultimate_df

In [ ]:
from src.data.build_ultimate_df import build_ultimate_df

ultimate_df = build_ultimate_df()

pd.set_option("display.float_format", "{:,.4f}".format)

DEFAULT_Y_COL_MAP = {
    "eur": "EURUSD",
    "gbp": "GBPUSD",
    "jpy": "USDJPY",
    "chf": "USDCHF",
    "cad": "USDCAD",
    "aud": "AUDUSD",
    "nzd": "NZDUSD",
    "nok": "USDNOK",
    "sek": "USDSEK",
}

def latest_window_multivariate_ols(df: pd.DataFrame, y_col: str, window: int = 252):
    """Fit multivariate OLS on the most recent rolling window and return (summary, r2)."""
    if y_col not in df.columns:
        raise KeyError(f"{y_col} not found in dataframe columns.")

    window_df = df.tail(window).dropna()
    if len(window_df) < 2:
        raise ValueError("Not enough non-NaN rows in the latest rolling window.")

    y = window_df[y_col]
    X = window_df.drop(columns=[y_col])
    X_const = sm.add_constant(X, has_constant="add")

    model = sm.OLS(y, X_const).fit()
    r2 = model.rsquared

    betas = model.params.drop("const")
    pvals = model.pvalues.drop("const")
    signif_pct = (1 - pvals) * 100

    ranked_idx = betas.abs().sort_values(ascending=False).index
    betas = betas.reindex(ranked_idx)
    signif_pct = signif_pct.reindex(ranked_idx)

    summary = pd.DataFrame({
        "Driver Name": betas.index,
        "Beta Coefficient": betas.values,
        "Rank": np.arange(1, len(betas) + 1),
        "Significance %": signif_pct.values,
    })
    summary["Beta Coefficient"] = summary["Beta Coefficient"].round(4)
    summary["Significance %"] = summary["Significance %"].round(2)
    return summary, r2

def run_processed_df_regression(df_name: str, y_col: str | None = None, window: int = 252):
    """Convenience wrapper for ultimate_df by name."""
    if df_name not in ultimate_df:
        raise KeyError(f"{df_name} not found in ultimate_df.")

    if y_col is None:
        y_col = DEFAULT_Y_COL_MAP.get(df_name)
        if y_col is None:
            raise KeyError(f"No default y_col found for {df_name}.")

    summary, r2 = latest_window_multivariate_ols(ultimate_df[df_name], y_col, window=window)
    display(summary)
    print(f"Current 1Y Rolling R2: {r2:.4f}")
    return summary, r2




In [ ]:
def run_processed_df_regression_sig_only(
    df_name: str,
    y_col: str | None = None,
    window: int = 252,
    min_significance: float = 95.0,
):
    """Same as run_processed_df_regression but filters to >= min_significance."""
    summary, r2 = run_processed_df_regression(df_name, y_col=y_col, window=window)
    filtered = summary[summary["Significance %"] >= min_significance].reset_index(drop=True)
    display(filtered)
    print(f"Current 1Y Rolling R2: {r2:.4f}")
    return filtered, r2




In [ ]:
run_processed_df_regression_sig_only("jpy")

In [ ]:
from src.ols_regressions import run_processed_df_regression_sig_only

# Now you can use the function:
run_processed_df_regression_sig_only("jpy")

In [ ]:
from src.ols_exBBDXY import run_processed_df_regression_sig_only

# Now you can use the function:
run_processed_df_regression_sig_only("aud")

In [4]:
from src.rolling_univariate_ols import build_rolling_maps
from src.data.build_ultimate_df import build_ultimate_df

ultimate_df = build_ultimate_df()
betas_map, signif_map = build_rolling_maps(ultimate_df, window=250)

# Example
nok_betas = betas_map["nok"]
nok_signif = signif_map["nok"]


c:\Users\tilik\Cambridge\Year_1\D200\fx_valuation\fxvaluation_d200_am3483\src\data\load_excel_sheets.py:85: UserWarning: Unable to parse some dates in sheet 'FX', column 'USDNOK Curncy'; those rows will be NaT
  warnings.warn(
c:\Users\tilik\Cambridge\Year_1\D200\fx_valuation\fxvaluation_d200_am3483\src\data\load_excel_sheets.py:85: UserWarning: Unable to parse some dates in sheet 'FX', column 'USDKRW Curncy'; those rows will be NaT
  warnings.warn(
c:\Users\tilik\Cambridge\Year_1\D200\fx_valuation\fxvaluation_d200_am3483\src\data\load_excel_sheets.py:85: UserWarning: Unable to parse some dates in sheet 'FX', column 'USDSGD Curncy'; those rows will be NaT
  warnings.warn(
c:\Users\tilik\Cambridge\Year_1\D200\fx_valuation\fxvaluation_d200_am3483\src\data\load_excel_sheets.py:85: UserWarning: Unable to parse some dates in sheet 'FX', column 'USDTWD Curncy'; those rows will be NaT
  warnings.warn(
c:\Users\tilik\Cambridge\Year_1\D200\fx_valuation\fxvaluation_d200_am3483\src\data\load_exce

In [5]:
nok_betas.tail()

,EMFX,2y yield,5y yield,10y yield,Real 2y yield,Local - S&P500,Local - Wilshire,Local - Dow Jones,MSCI World - S&P500,MSCI World - Wilshire,...,Gold,Oil,Copper COMEX,Copper LME,MOVE Index,JPMVG71M Index,NG1 COMB Comdty,TZT1 Comdty,FN1 Comdty,VIX Index
2026-02-13,-1.555874,2.718668,2.585089,2.573896,-1.175464,0.033115,0.034016,0.017577,-0.237418,-0.190112,...,-0.200404,-0.084067,-0.090635,-0.182879,-0.000688,-0.026264,-0.076457,-0.012956,-0.006451,0.011269
2026-02-16,-1.556633,2.688309,2.545148,2.528801,-1.200536,0.038205,0.039170,0.022607,-0.228002,-0.180017,...,-0.199404,-0.085546,-0.090222,-0.182296,NaN,-0.025258,NaN,-0.013290,-0.006569,0.011269
2026-02-17,-1.552223,2.838352,2.811605,2.850888,-1.250758,0.034916,0.035993,0.019293,-0.228975,-0.180729,...,-0.197915,-0.086627,-0.090994,-0.184264,-0.000747,-0.023490,-0.079543,-0.014684,-0.006498,0.011081
2026-02-18,-1.543528,2.697706,2.673176,2.724490,-1.192289,0.032493,0.033606,0.016245,-0.203022,-0.157784,...,-0.198124,-0.091507,-0.087973,-0.178405,NaN,-0.018060,-0.076714,-0.017336,-0.007375,0.011070
NaT,-1.543433,2.702999,2.671920,2.717020,-1.184226,0.034231,0.035406,0.017343,-0.200185,-0.154588,...,-0.198417,-0.091443,-0.087954,-0.178349,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
nok_signif.tail()

,EMFX,2y yield,5y yield,10y yield,Real 2y yield,Local - S&P500,Local - Wilshire,Local - Dow Jones,MSCI World - S&P500,MSCI World - Wilshire,...,Gold,Oil,Copper COMEX,Copper LME,MOVE Index,JPMVG71M Index,NG1 COMB Comdty,TZT1 Comdty,FN1 Comdty,VIX Index
2026-02-13,100.0,99.988298,99.967822,99.881348,99.547771,72.851914,74.821565,40.769729,96.663329,92.671138,...,100.0,99.996290,99.999999,100.0,23.154209,58.999519,80.344544,80.623649,86.956110,86.087899
2026-02-16,100.0,99.985826,99.956084,99.828980,99.314300,79.074339,80.830790,50.444192,95.683049,90.624564,...,100.0,99.997361,99.999999,100.0,NaN,57.239147,NaN,81.216496,86.947029,86.151094
2026-02-17,100.0,99.994664,99.988852,99.954849,99.521724,74.846997,76.940961,43.926906,95.833570,90.848815,...,100.0,99.998117,100.000000,100.0,25.246698,53.996282,82.188087,85.295472,85.871069,85.586440
2026-02-18,100.0,99.988619,99.977738,99.924932,99.325709,71.965814,74.238757,38.013493,93.343261,86.618672,...,100.0,99.999568,99.999999,100.0,NaN,43.295765,80.563635,90.965426,89.944339,85.524381
NaT,100.0,99.987364,99.975239,99.916818,99.268121,74.229003,76.459080,40.282513,92.823133,85.579930,...,100.0,99.999564,99.999999,100.0,NaN,NaN,NaN,NaN,NaN,NaN
